In [8]:
%load_ext autoreload
%autoreload 2

import os
import sys
import sqlite3
import pprint
from openai import OpenAI
from dotenv import load_dotenv  # Gestione del file .
from auditor_core import scarica_ultimo_log, valuta_chat_con_gemma, salva_valutazione_db


# Carica esplicitamente il file .env posizionato nella radice del progetto (un livello sopra 'notebooks')
load_dotenv(os.path.join("..", ".env"))

# Risoluzione dinamica del percorso per importare i moduli dalla cartella 'src'
sys.path.append(os.path.abspath(os.path.join("..", "src")))


# RECUPERO VARIABILI D'AMBIENTE (.env)
API_KEY_HERO = os.getenv("PLATFORMHERO_TOKEN")
URL_API_HERO = os.getenv("PLATFORMHERO_URL")  # Endpoint per scaricamento log di produzione


USA_MODELLO_LOCALE = True  # Imposta su False se vuoi usare l'API Cloud di PlatformHero

if USA_MODELLO_LOCALE:
    
    MODEL_NAME = "llama3"
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama-local-dummy-key"
else:
    
    MODEL_NAME = "gpt-4o-mini"  
    BASE_URL = "https://api.platformhero.ai/v1"
    API_KEY = API_KEY_HERO  # Utilizza automaticamente il token recuperato dal file .env


client_openai = OpenAI(base_url=BASE_URL, api_key=API_KEY)


DB_PATH = os.path.join("..", "data", "platformhero_mirror.db")

def mia_connessione():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn

# BLOCCO DI DIAGNOSTICA STATO ENVIRONMENT 
if not API_KEY_HERO:
    print("ATTENZIONE: Il token PLATFORMHERO_TOKEN è VUOTO o non viene letto dal file .env!")
else:
    print(f"Token PlatformHero individuato! Lunghezza: {len(API_KEY_HERO)} caratteri.")

print(f"Sistema pronto! Modello configurato: {MODEL_NAME} (Locale: {USA_MODELLO_LOCALE})")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Token PlatformHero individuato! Lunghezza: 231 caratteri.
Sistema pronto! Modello configurato: llama3 (Locale: True)


In [9]:


#  Scarica il log già filtrato dal core (prenderà quello con l'array 'messages')
log_reale = scarica_ultimo_log(API_KEY_HERO, URL_API_HERO)
id_del_log = log_reale.get("log_id", "ID_Mancante")

# Estrazione ricorsiva della cronologia messaggi
lista_messaggi = log_reale.get("content", {}).get("messages", [])

# Formattazione della chat in un'unica stringa per Llama 3
testo_ticket = ""
for msg in lista_messaggi:
    ruolo = msg.get("role", "utente").upper()
    contenuto = msg.get("content", "")
    testo_ticket += f"--- {ruolo} ---\n{contenuto}\n\n"

print(" Log agganciato correttamente")
print(f"f ID Log: {id_del_log}")
print(f"f Caratteri estratti per la valutazione: {len(testo_ticket)}")

 Log agganciato correttamente
f ID Log: HjV3kuq7EXR6qHHoSoIEWj4z
f Caratteri estratti per la valutazione: 16817


In [10]:
risposta_ai = valuta_chat_con_gemma(client_openai, MODEL_NAME, testo_ticket)

# 2. Stampa l'output testuale direttamente nel notebook
print("=== 🤖 ECCO COSA HA SCRITTO L'LLM ===")
print(risposta_ai)
print("=====================================")

salva_valutazione_db(mia_connessione, id_del_log, risposta_ai)

=== 🤖 ECCO COSA HA SCRITTO L'LLM ===
{
"system_judicial_reasoning": "Valutazione tecnica ottima: correttezza dei comandi, sintassi di codice e query SQL. Tono professionale, assenza di violazioni sicurezza.",
"technical_score": 9.0,
"completeness_score": 10.0,
"business_score": 9.5,
"consistency_score": 9.5,
"overall_score": 9.3,
"feedback": "Valutazione esemplare: tecnica, resa, tone.",
"issues": ""
}
 Successo! Valutazione registrata. Voto LLM: 9.3/10.0


In [11]:

print("EFFETTIVA STRUTTURA DEL CAMPO 'CONTENT':")
print("-" * 60)
# Ispezioniamo i dati reali del log appena scaricato
pprint.pprint(log_reale.get("content"))
print("-" * 60)

EFFETTIVA STRUTTURA DEL CAMPO 'CONTENT':
------------------------------------------------------------
{'functions': [],
 'messages': [{'content': 'You are a support ticket quality evaluator. Your '
                          'sole task is to analyze a support ticket '
                          'conversation and produce a single JSON score.\n'
                          '\n'
                          '## Situation\n'
                          'You are embedded in an automated quality assurance '
                          'pipeline. You receive the full text of a closed '
                          'support ticket conversation between a customer and '
                          'support agent(s). Your output feeds directly into a '
                          'reporting dashboard — no human will read your '
                          'reasoning, only the JSON score.\n'
                          '\n'
                          '## Task\n'
                          'Analyze the conversation and pr